In [1]:
"""
extract_dataset.py

Reads list_attr_celeba.csv, filters images that exist in img_align_celeba/,
copies them to a new folder (celeba_cgan/) and saves a clean CSV with only
the 3 attributes we need: Male, Young, Smiling.

Run this once before training the cGAN.
"""

import shutil
from pathlib import Path

import pandas as pd
from tqdm import tqdm

# ================= CONFIG =================
DATASET_DIR = Path(r"C:\Users\arfao\Downloads\celeba_dataset")
ATTR_CSV    = DATASET_DIR / "list_attr_celeba.csv"
SOURCE_DIR  = DATASET_DIR / "img_align_celeba" / "img_align_celeba"
OUTPUT_DIR  = Path(r"C:\Users\arfao\Desktop\data_preprocessing_celeba\celeba_cgan")
OUTPUT_CSV  = Path(r"C:\Users\arfao\Desktop\data_preprocessing_celeba\celeba_cgan_labels.csv")

ATTRIBUTES  = ["Male", "Young", "Smiling"]

# ================= LOAD ATTRIBUTES =================
print("Loading attribute file...")
df = pd.read_csv(ATTR_CSV)

# strip quotes from column names
df.columns = [c.strip().strip("'\"") for c in df.columns]

# strip quotes from image_id values
df["image_id"] = df["image_id"].str.strip().str.strip("'\"")

# keep only the columns we need
df = df[["image_id"] + ATTRIBUTES]

# convert -1/1 to 0/1
for col in ATTRIBUTES:
    df[col] = df[col].apply(lambda x: 1 if int(x) == 1 else 0)

print(f"Loaded {len(df)} rows from CSV")
print(f"Sample:\n{df.head(2).to_string()}")

# ================= FILTER TO EXISTING IMAGES =================
print(f"\nChecking which images exist in {SOURCE_DIR}...")

valid_rows = []
missing = 0

for _, row in tqdm(df.iterrows(), total=len(df)):
    src_path = SOURCE_DIR / row["image_id"]
    if src_path.exists():
        valid_rows.append(row)
    else:
        missing += 1

df_filtered = pd.DataFrame(valid_rows).reset_index(drop=True)

print(f"\nFound   : {len(df_filtered)} images")
print(f"Missing : {missing} images not found in folder")

if len(df_filtered) == 0:
    print("\nERROR: No images matched. Check SOURCE_DIR path above.")
    exit()

# ================= SHOW DISTRIBUTION =================
print("\nAttribute distribution:")
for col in ATTRIBUTES:
    count_1 = df_filtered[col].sum()
    count_0 = len(df_filtered) - count_1
    label_1 = "Male"   if col == "Male" else col
    label_0 = "Female" if col == "Male" else f"Not {col}"
    print(f"  {col:8}: {label_1}={count_1} | {label_0}={count_0}")

print("\nCombination distribution (8 combinations):")
combo_counts = df_filtered.groupby(ATTRIBUTES).size().reset_index(name="count")
combo_counts["Male_label"]    = combo_counts["Male"].apply(lambda x: "Male"    if x == 1 else "Female")
combo_counts["Young_label"]   = combo_counts["Young"].apply(lambda x: "Young"  if x == 1 else "Old")
combo_counts["Smiling_label"] = combo_counts["Smiling"].apply(lambda x: "Smiling" if x == 1 else "Neutral")
for _, row in combo_counts.iterrows():
    print(f"  {row['Male_label']:8} | {row['Young_label']:8} | {row['Smiling_label']:8} => {row['count']} images")

# ================= COPY IMAGES =================
print(f"\nCopying images to {OUTPUT_DIR}...")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for _, row in tqdm(df_filtered.iterrows(), total=len(df_filtered)):
    src = SOURCE_DIR / row["image_id"]
    dst = OUTPUT_DIR / row["image_id"]
    if not dst.exists():
        shutil.copy2(src, dst)

# ================= SAVE LABELS CSV =================
df_filtered.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved labels to {OUTPUT_CSV}")
print(f"Saved {len(df_filtered)} images to {OUTPUT_DIR}/")
print("\nDone! You can now run train_cgan.py")

Loading attribute file...
Loaded 202599 rows from CSV
Sample:
     image_id  Male  Young  Smiling
0  000001.jpg     0      1        1
1  000002.jpg     0      1        1

Checking which images exist in C:\Users\arfao\Downloads\celeba_dataset\img_align_celeba\img_align_celeba...


100%|██████████| 202599/202599 [00:20<00:00, 10074.59it/s]



Found   : 202599 images
Missing : 0 images not found in folder

Attribute distribution:
  Male    : Male=84434 | Female=118165
  Young   : Young=156734 | Not Young=45865
  Smiling : Smiling=97669 | Not Smiling=104930

Combination distribution (8 combinations):
  Female   | Old      | Neutral  => 4687 images
  Female   | Old      | Smiling  => 10191 images
  Female   | Young    | Neutral  => 49607 images
  Female   | Young    | Smiling  => 53680 images
  Male     | Old      | Neutral  => 17661 images
  Male     | Old      | Smiling  => 13326 images
  Male     | Young    | Neutral  => 32975 images
  Male     | Young    | Smiling  => 20472 images

Copying images to C:\Users\arfao\Desktop\data_preprocessing_celeba\celeba_cgan...


100%|██████████| 202599/202599 [03:37<00:00, 933.12it/s] 



Saved labels to C:\Users\arfao\Desktop\data_preprocessing_celeba\celeba_cgan_labels.csv
Saved 202599 images to C:\Users\arfao\Desktop\data_preprocessing_celeba\celeba_cgan/

Done! You can now run train_cgan.py
